In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

sys.path.append('..')
sys.path.append(os.path.abspath(os.path.join('..', 'magnet-pinn')))
sys.path.append(os.path.abspath(os.path.join('..', 'neuraloperator')))

In [3]:
from torch.utils.data import DataLoader

from magnet_pinn.data.transforms import Compose, Crop, GridPhaseShift
from magnet_pinn.data.grid import MagnetGridIterator

TRAIN_DIR = "../data/processed/batch_17/grid_voxel_size_4_data_type_float32"
VAL_DIR = "../data/processed/batch_17/grid_voxel_size_4_data_type_float32"

augmentation = Compose(
    [
        Crop(crop_size=(100, 100, 100)),
        GridPhaseShift(num_coils=8)
    ]
)

val_set = MagnetGridIterator(VAL_DIR, transforms=augmentation, num_samples=8)
val_loader = iter(DataLoader(val_set, batch_size=1))

In [ ]:
import pytorch_lightning as pl

from torch.utils.data import DataLoader

from magnet_pinn.utils import StandardNormalizer, StandardNormalizerSqrt
from magnet_pinn.data.transforms import Compose, Crop, GridPhaseShift
from magnet_pinn.data.grid import MagnetGridIterator
from magnet_pinn.data.utils import worker_init_fn

from neuralop.models import FNO, UNO
from mrifield.train.lit_mrifield import LitMRIField
from mrifield.models import AFNONet, FNOFactorizedMesh3D, UFNO3D, CNO
from magnet_pinn.losses.physics import DivergenceLoss

#model = FNO(n_modes=(16, 16, 16), in_channels=5, out_channels=12, hidden_channels=42)
#model = FNO(n_modes=(16, 16, 16), in_channels=5, out_channels=12, hidden_channels=59, factorization='tucker', rank=0.1)
#model = UNO(in_channels=5, out_channels=12, hidden_channels=16, n_layers=4, uno_out_channels=[32,64,64,32], uno_n_modes=[[13,13,13],[13,13,13],[13,13,13],[13,13,13]], uno_scalings=[[1,1,1],[0.5,0.5,0.5],[1,1,1],[2,2,2]], channel_mlp_skip='linear')
#model = UNO(in_channels=5, out_channels=12, hidden_channels=16, n_layers=7, uno_out_channels=[16,32,64,128,64,32,16], uno_n_modes=[[13,13,13],[13,13,13],[13,13,13],[13,13,13],[13,13,13],[13,13,13],[13,13,13]], uno_scalings=[[1,1,1],[0.5,0.5,0.5],[0.5,0.5,0.5],[1,1,1],[1,1,1],[2,2,2],[2,2,2]], channel_mlp_skip='linear')
#model = AFNONet(depth=5)
#model = FNOFactorizedMesh3D(modes_x=32, modes_y=32, modes_z=32, width=42, input_dim=5, output_dim=12, n_layers=4, share_weight=False, factor=4, ff_weight_norm=False, n_ff_layers=2, layer_norm=True)
#model = UFNO3D(in_channels=12, out_channels=1, modes1=16, modes2=16, modes3=16, width=32)
model = CNO3d(in_dim=5, out_dim=12, size=100, N_layers=4, channel_multiplier=26)

input_normalizer = StandardNormalizer.load_from_json(f"{TRAIN_DIR}/normalization/std/input_normalization.json")
target_normalizer = StandardNormalizerSqrt.load_from_json(f"{TRAIN_DIR}/normalization/std/target_normalization.json")

augmentation = Compose(
    [
        Crop(crop_size=(100, 100, 100)),
        GridPhaseShift(num_coils=8)
    ]
)

lit_model = LitMRIField(model, input_normalizer, target_normalizer)

train_set = MagnetGridIterator(TRAIN_DIR, transforms=augmentation, num_samples=100)
val_set = MagnetGridIterator(VAL_DIR, transforms=augmentation, num_samples=100)

train_loader = DataLoader(train_set, batch_size=4, num_workers=4, worker_init_fn=worker_init_fn)
val_loader = DataLoader(val_set, batch_size=4, num_workers=4, worker_init_fn=worker_init_fn)

trainer = pl.Trainer(accelerator="cpu", devices=1, log_every_n_steps=100, max_epochs=10)
trainer.fit(model=lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

/Users/mariusbohn/Documents/Studium/MA/ma_bohn/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/mariusbohn/Documents/Studium/MA/ma_bohn/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


ImportError: cannot import name 'CNO3d' from 'mrifield.models' (/Users/mariusbohn/Documents/Studium/MA/ma_bohn/results/../mrifield/models/__init__.py)